[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/sodacore-certified/notebooks/day-08-schema-evolution.ipynb#scrollTo=1a2b3c4d)

---
# Day 8 · Schema Evolution — Detecting and Handling Upstream Schema Changes
**certified-journeys / sodacore-certified** · Review · Schema checks, column diffs, and break vs. bend

> **Goal for today:** Write schema checks that enforce required columns, data types, and order — then simulate both breaking and non-breaking schema changes and extract a structured column diff from scan results.


In [ ]:
%pip install -q soda-core-duckdb


## Step 1 · Why Schema Evolution Breaks Pipelines

Upstream teams change table schemas more often than data teams expect. The three most common changes:

| Change type | Example | Impact |
|---|---|---|
| Column dropped | `amount` removed | **Breaking** — downstream reads fail |
| Column renamed | `amt` → `amount` | **Breaking** — silent wrong-column reads |
| Column added | New `discount` column | **Non-breaking** — downstream unaffected |
| Type widened | `INTEGER` → `BIGINT` | Usually safe; depends on consumer |
| Type narrowed | `DOUBLE` → `INTEGER` | **Breaking** — precision loss |

Soda Core's `schema:` check lets you declare **exactly** what the table must look like and raise FAIL or WARN on any deviation. The key distinction:

- `fail:` → scan outcome is FAIL → pipeline should stop
- `warn:` → scan outcome is WARN → pipeline can continue with a notification


In [ ]:
import duckdb, tempfile, pathlib
from soda.scan import Scan

# ── Create a baseline orders table ────────────────────────────────────────────
tmpdir = pathlib.Path(tempfile.mkdtemp())
db_path = str(tmpdir / "schema_test.duckdb")

conn = duckdb.connect(db_path)
conn.execute("""
    CREATE TABLE orders (
        id          INTEGER,
        customer_id INTEGER,
        amount      DOUBLE,
        status      VARCHAR
    )
""")
conn.execute("INSERT INTO orders VALUES (1, 101, 49.99, 'completed')")
conn.execute("INSERT INTO orders VALUES (2, 102, 120.0, 'pending')")
conn.close()

config_yml = f"""
data_sources:
  mydb:
    type: duckdb
    path: "{db_path}"
"""
config_path = tmpdir / "configuration.yml"
config_path.write_text(config_yml)
print("Baseline database ready at:", db_path)


**What just happened?**

- We created a DuckDB table with four columns: `id`, `customer_id`, `amount`, `status`.
- The config file maps `mydb` to this database — used by every scan in this notebook.
- All subsequent steps mutate the schema and re-run checks against the same DuckDB file.


## Step 2 · Writing a Comprehensive Schema Check

The SodaCL `schema:` check supports three sub-conditions:

```yaml
checks for orders:
  - schema:
      fail:
        when required column missing: [id, customer_id, amount, status]
        when forbidden column present: [ssn, credit_card_number]
      warn:
        when wrong column type:
          amount: double
          id: integer
        when wrong column index:
          id: 0
          customer_id: 1
```

- **`when required column missing`** — FAIL if any listed column is absent
- **`when forbidden column present`** — FAIL if a sensitive column appears (PII guard)
- **`when wrong column type`** — WARN/FAIL if type doesn't match
- **`when wrong column index`** — WARN/FAIL if column position changed (matters for positional CSV readers)


In [ ]:
def run_schema_check(db_path: str, config_path: pathlib.Path, tmpdir: pathlib.Path,
                     label: str = "schema check") -> dict:
    """Run the standard schema check and return a summary dict."""
    checks_yml = """
checks for orders:
  - schema:
      fail:
        when required column missing: [id, customer_id, amount, status]
      warn:
        when wrong column type:
          amount: double
          id: integer
        when wrong column index:
          id: 0
          customer_id: 1
"""
    checks_path = tmpdir / "schema_checks.yml"
    checks_path.write_text(checks_yml)

    scan = Scan()
    scan.set_data_source_name("mydb")
    scan.add_configuration_yaml_file(str(config_path))
    scan.add_sodacl_yaml_file(str(checks_path))
    scan.execute()

    outcome = "PASS"
    if scan.has_check_fails():
        outcome = "FAIL"
    elif scan.has_check_warns():
        outcome = "WARN"

    print(f"\n{'='*50}")
    print(f"Scenario: {label}")
    print(f"Outcome:  {outcome}")
    print("Logs:")
    print(scan.get_logs_text())
    return {"label": label, "outcome": outcome, "logs": scan.get_logs_text()}


# ── Baseline: all columns present — expect PASS ────────────────────────────
result_baseline = run_schema_check(db_path, config_path, tmpdir, label="Baseline (all columns present)")


**What just happened?**

- The baseline scan passes because `orders` has all four required columns with the correct types.
- `scan.has_check_fails()` and `scan.has_check_warns()` are the two boolean helpers you use in CI to gate pipeline execution.
- **`run_schema_check` is reused** for all three scenarios in this notebook — only the database state changes.


## Step 3 · Simulate a Breaking Schema Change (Drop a Column)

We simulate an upstream team removing the `amount` column. This is a **breaking change** — any downstream `SELECT amount FROM orders` will fail, and our quality gate should catch it immediately.

In a real data warehouse, you'd compare schema snapshots captured between pipeline runs. Here we `ALTER TABLE` to mutate the live table and re-run the same schema check.


In [ ]:
# ── Breaking change: drop the 'amount' column ──────────────────────────────
conn = duckdb.connect(db_path)
conn.execute("ALTER TABLE orders DROP COLUMN amount")
print("Remaining columns:", [row[0] for row in conn.execute("DESCRIBE orders").fetchall()])
conn.close()

result_breaking = run_schema_check(db_path, config_path, tmpdir, label="Breaking change: amount column dropped")
print("\nExpected outcome: FAIL")
print("Actual outcome:  ", result_breaking["outcome"])


**What just happened?**

- `ALTER TABLE orders DROP COLUMN amount` removes `amount` from the live DuckDB table.
- The schema check fires `when required column missing: [amount]` and the scan outcome is **FAIL**.
- **In a pipeline:** `scan.has_check_fails()` returns `True` → raise an exception → DAG run fails → alert fires.
- The FAIL is intentional and informative — the logs will explicitly name the missing column.


## Step 4 · Simulate a Non-Breaking Change (Add a Column)

Adding a column is generally **non-breaking** — existing consumers ignore unknown columns. We restore `amount`, then add a new `discount` column and verify the check still passes.

Non-breaking changes should produce a **WARN** (alerting without stopping the pipeline) if you want visibility. Achieve this by adding the new column to the `when forbidden column present` list during a transition period.


In [ ]:
# ── Restore amount, then add new column discount ──────────────────────────
conn = duckdb.connect(db_path)
conn.execute("ALTER TABLE orders ADD COLUMN amount DOUBLE")
conn.execute("UPDATE orders SET amount = 49.99 WHERE id = 1")
conn.execute("UPDATE orders SET amount = 120.0 WHERE id = 2")
conn.execute("ALTER TABLE orders ADD COLUMN discount DOUBLE DEFAULT 0.0")
print("Columns after adding discount:",
      [row[0] for row in conn.execute("DESCRIBE orders").fetchall()])
conn.close()

result_nonbreaking = run_schema_check(db_path, config_path, tmpdir,
                                       label="Non-breaking change: discount column added")
print("\nExpected outcome: PASS (extra column is not forbidden)")
print("Actual outcome:  ", result_nonbreaking["outcome"])


**What just happened?**

- `ALTER TABLE orders ADD COLUMN discount DOUBLE` is a **non-breaking additive change** — it passes because the schema check only looks for missing required columns.
- The scan outcome remains **PASS** — downstream pipelines continue unaffected.
- To turn this into a WARN (so teams are aware of the new column), add `discount` to a `when forbidden column present` list temporarily, then remove it after the pipeline is updated.
- **Design principle:** use FAIL for anything that breaks consumption; use WARN for changes that need attention but don't stop the pipeline.


## Step 5 · Extracting the Column Diff from Scan Results

When a schema check fires, you want a structured diff — not just a log message. The `scan.get_scan_results()` method returns a dictionary with a `checks` list; each check entry includes diagnostic details for schema checks.

We write `extract_column_diff(scan)` to parse those results and return a structured dict with `missing`, `unexpected`, and `wrong_type` lists.


In [ ]:
import re

def extract_column_diff(scan: Scan) -> dict:
    """
    Parse scan results and return a structured column diff.
    Returns: {"missing": [...], "unexpected": [...], "wrong_type": [...], "wrong_index": [...]}
    """
    diff = {"missing": [], "unexpected": [], "wrong_type": [], "wrong_index": []}
    logs = scan.get_logs_text()

    # Parse from log text — schema check logs include structured column information
    # Pattern for missing columns: lines containing 'required' and 'missing'
    for line in logs.splitlines():
        line_lower = line.lower()
        if "required column" in line_lower and "missing" in line_lower:
            # Extract column name(s) from brackets if present
            matches = re.findall(r"'(\w+)'", line)
            diff["missing"].extend(matches)
        elif "wrong column type" in line_lower:
            matches = re.findall(r"'(\w+)'", line)
            diff["wrong_type"].extend(matches)
        elif "wrong column index" in line_lower:
            matches = re.findall(r"'(\w+)'", line)
            diff["wrong_index"].extend(matches)
        elif "forbidden" in line_lower and "present" in line_lower:
            matches = re.findall(r"'(\w+)'", line)
            diff["unexpected"].extend(matches)

    return diff


# ── Demonstrate on a fresh breaking-change scan ─────────────────────────────
conn = duckdb.connect(db_path)
conn.execute("ALTER TABLE orders DROP COLUMN status")   # drop another required column
conn.close()

checks_yml = """
checks for orders:
  - schema:
      fail:
        when required column missing: [id, customer_id, amount, status]
      warn:
        when wrong column type:
          amount: double
          id: integer
"""
checks_path = tmpdir / "schema_checks.yml"
checks_path.write_text(checks_yml)

diff_scan = Scan()
diff_scan.set_data_source_name("mydb")
diff_scan.add_configuration_yaml_file(str(config_path))
diff_scan.add_sodacl_yaml_file(str(checks_path))
diff_scan.execute()

diff = extract_column_diff(diff_scan)
print("Column diff result:")
print(f"  Missing columns:   {diff['missing']}")
print(f"  Unexpected columns:{diff['unexpected']}")
print(f"  Wrong type:        {diff['wrong_type']}")
print(f"  Wrong index:       {diff['wrong_index']}")
print()
print("Full scan logs:")
print(diff_scan.get_logs_text())


**What just happened?**

- `extract_column_diff` parses Soda's log output using regex — this is the lightweight approach that works without a third-party YAML parser.
- The diff dict is structured so you can pass it to a Slack message formatter, write it to a metadata store, or use it to generate a migration script automatically.
- **Production enhancement:** write the diff to a Delta table or a `schema_drift_events` table with `timestamp`, `table_name`, and the diff JSON so you have a historical record of schema changes.


## Step 6 · Schema Evolution Strategy Matrix

| Change | Check outcome | Recommended action |
|---|---|---|
| Required column missing | FAIL | Stop pipeline immediately; page on-call |
| Column added (unknown) | PASS or WARN | Log and notify; update contract |
| Column type narrowed | FAIL or WARN | Stop pipeline; assess precision loss |
| Column type widened | WARN | Continue with notification |
| Column renamed | FAIL (missing + unexpected) | Stop pipeline; co-ordinate rename migration |
| Column reordered | WARN (wrong index) | Continue; notify positional readers |

The distinction between FAIL and WARN in your SodaCL checks **encodes your team's tolerance** for each change type. Treat the schema check YAML as a living contract — update it via pull request whenever the schema contract changes legitimately.

> **Tip:** Run schema checks at the **start** of every pipeline run, before any transformation. Catching schema drift early is cheaper than discovering it after a 2-hour dbt run.


In [ ]:
# Challenge: Extend extract_column_diff to also return the actual schema
# seen by Soda (columns present in the table) vs. the expected schema.
#
# Hint: After running a scan, use DuckDB to query DESCRIBE orders and
# compare the result to your expected columns list.
#
# Scaffold:

def full_schema_diff(scan: Scan, db_path: str, expected_columns: list) -> dict:
    """
    Return a comprehensive diff:
      actual_columns: list from DESCRIBE
      missing:        columns in expected_columns but not in actual
      added:          columns in actual but not in expected_columns
      soda_diff:      result from extract_column_diff(scan)
    """
    # 1. Query actual columns via DuckDB
    # conn = duckdb.connect(db_path); rows = conn.execute("DESCRIBE orders").fetchall()
    # actual = [row[0] for row in rows]; conn.close()

    # 2. Compute set differences
    # missing = [c for c in expected_columns if c not in actual]
    # added   = [c for c in actual if c not in expected_columns]

    # 3. Call extract_column_diff(scan) and merge
    pass


---
## Day 8 key concepts recap
| Concept | What to remember |
|---|---|
| `schema:` check | Declares required columns, forbidden columns, types, and positions |
| `fail: when required column missing` | Stops the pipeline when a contract column disappears |
| `warn: when wrong column type` | Alerts without stopping when type drifts |
| Breaking vs non-breaking | Drop/rename = breaking; add = non-breaking |
| `scan.has_check_fails()` | Boolean gate for CI/CD pipeline decisions |
| Column diff extraction | Parse `get_logs_text()` with regex to get structured missing/unexpected/wrong-type lists |

> **Tip:** Run schema checks at pipeline ingestion time — not after transformations. The earlier you catch drift, the less compute you waste.

---
## What's next
**Day 9** → Test-Driven Data Development — writing checks before transformations, red-green workflow, and quality gate matrices.

Mark Day 8 complete in your [tracker](../index.html).
